# Persecucion e intercepcion de balon con Monte Carlo Control

Curso DS5345 Aprendizaje por Refuerzo. Tarea acotada Ball Pursuit en RoboCup 2D.
Algoritmo Monte Carlo Control On Policy First Visit con comparacion de exploracion.

Contenido del cuaderno. Imports. Formulacion MDP. Entorno y discretizacion. Monte Carlo First Visit. Entrenamiento con epsilon constante. Entrenamiento con epsilon decreciente. Comparacion de G0. Comparacion de tasa de exito. Comparacion de pasos. Funcion de valor V. Politica pi. Trayectoria 2D. Conexion con rcssserver. Conclusiones.

In [ ]:
import math
import time
import os
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
np.random.seed(42)
if os.path.isdir("/workspace/results"):
    OUT = "/workspace/results"
elif os.path.isdir("../results"):
    OUT = "../results"
else:
    OUT = "results"
os.makedirs(OUT, exist_ok=True)
print(f"Resultados en {OUT}.")

## 1 Formulacion MDP de Ball Pursuit

Tupla S A P R gamma.

S es la posicion relativa al balon en polares discretizadas. Distancia d_b en 4 zonas. Zona 0 menor a 0.8 m que es captura. Zona 1 de 0.8 a 3 m. Zona 2 de 3 a 8 m. Zona 3 mayor a 8 m. Angulo theta_b en 5 sectores. Frontal si valor absoluto menor o igual a 15. Derecha frontal de menos 60 a menos 15. Izquierda frontal de 15 a 60. Atras derecha menor a menos 60. Atras izquierda mayor a 60. Total 4 por 5 igual a 20 estados.

A son 4 macro acciones. 0 es DASH 100. 1 es DASH 50. 2 es TURN mas 35. 3 es TURN menos 35.

P son las transiciones del movimiento y giro con la dinamica del simulador. El estado siguiente depende de la posicion y orientacion despues de avanzar o girar.

R es la recompensa oficial. Cambio de distancia menos 0.2 mas 100 si hay captura. En formula delta d_b menos 0.2 mas 100 por indicador de d_b menor o igual a 0.8.

Gamma es 0.99. Descuenta poco el futuro y conserva la senal del bono de captura.

La tabla Q tiene dimension 20 por 4 igual a 80 valores. V se obtiene como maximo sobre acciones. Pi se obtiene como argmax sobre acciones.

## 2 Entorno de persecucion y discretizacion

El jugador inicia cerca de x igual a menos 15 con orientacion aleatoria. El balon inicia cerca del centro. La distancia inicial queda entre 5 y 40 m. El episodio termina con captura si d_b menor a 0.8 o a los 40 pasos. La recompensa de cada paso es oficial.

In [ ]:
class BallPursuitSimEnv:
    def __init__(self, max_steps=40):
        self.max_steps = max_steps
        self.reset()
    def reset(self):
        self.player_x = -15.0 + np.random.uniform(-2.0, 2.0)
        self.player_y = np.random.uniform(-4.0, 4.0)
        self.player_theta = np.random.uniform(-180.0, 180.0)
        self.ball_x = 0.0 + np.random.uniform(-2.0, 2.0)
        self.ball_y = np.random.uniform(-3.0, 3.0)
        self.steps = 0
        self.trajectory_x = [self.player_x]
        self.trajectory_y = [self.player_y]
        return self._get_state()
    def _get_obs(self):
        dx = self.ball_x - self.player_x
        dy = self.ball_y - self.player_y
        dist = math.hypot(dx, dy)
        angle_global = math.degrees(math.atan2(dy, dx))
        angle_rel = (angle_global - self.player_theta + 180.0) % 360.0 - 180.0
        return dist, angle_rel
    def _discretize(self, dist, angle_rel):
        if dist < 0.8:
            d_bin = 0
        elif dist < 3.0:
            d_bin = 1
        elif dist < 8.0:
            d_bin = 2
        else:
            d_bin = 3
        if abs(angle_rel) <= 15.0:
            a_bin = 0
        elif -60.0 <= angle_rel < -15.0:
            a_bin = 1
        elif 15.0 < angle_rel <= 60.0:
            a_bin = 2
        elif -180.0 <= angle_rel < -60.0:
            a_bin = 3
        else:
            a_bin = 4
        return (d_bin, a_bin)
    def _get_state(self):
        dist, angle_rel = self._get_obs()
        return self._discretize(dist, angle_rel)
    def step(self, action):
        self.steps += 1
        dist_prev, _ = self._get_obs()
        if action == 0:
            rad = math.radians(self.player_theta)
            self.player_x += 1.0 * math.cos(rad)
            self.player_y += 1.0 * math.sin(rad)
        elif action == 1:
            rad = math.radians(self.player_theta)
            self.player_x += 0.5 * math.cos(rad)
            self.player_y += 0.5 * math.sin(rad)
        elif action == 2:
            self.player_theta = (self.player_theta + 35.0 + 180.0) % 360.0 - 180.0
        elif action == 3:
            self.player_theta = (self.player_theta - 35.0 + 180.0) % 360.0 - 180.0
        self.trajectory_x.append(self.player_x)
        self.trajectory_y.append(self.player_y)
        dist_curr, angle_curr = self._get_obs()
        done = False
        if dist_curr < 0.8:
            reward = (dist_prev - dist_curr) - 0.2 + 100.0
            done = True
        elif self.steps >= self.max_steps:
            reward = (dist_prev - dist_curr) - 0.2
            done = True
        else:
            reward = (dist_prev - dist_curr) - 0.2
        return self._discretize(dist_curr, angle_curr), reward, done, {"dist": dist_curr, "angle": angle_curr}
env_check = BallPursuitSimEnv()
ds = []
for _ in range(500):
    env_check.reset()
    d, _ = env_check._get_obs()
    ds.append(d)
print(f"Distancia inicial min {min(ds):.2f} max {max(ds):.2f} media {np.mean(ds):.2f}.")
print(f"Estados 20. Acciones 4. Q de 20 por 4.")

## 3 Monte Carlo Control On Policy First Visit

Se genera el episodio completo con epsilon greedy. Al final se recorre hacia atras con G igual a gamma por G mas r. Solo se actualiza la primera visita de cada par estado accion. Q es el promedio de retornos. La politica greedy es argmax de Q.

In [ ]:
def train_mc_control(n_episodes=3500, gamma=0.99, mode="const", eps_const=0.1, eps_start=1.0, eps_min=0.05, eps_decay=0.998, seed=42):
    np.random.seed(seed)
    n_actions = 4
    Q = defaultdict(lambda: np.zeros(n_actions, dtype=np.float64))
    returns_sum = defaultdict(lambda: np.zeros(n_actions, dtype=np.float64))
    returns_count = defaultdict(lambda: np.zeros(n_actions, dtype=np.int32))
    hist_g0 = []
    hist_steps = []
    hist_succ = []
    hist_eps = []
    for ep in range(n_episodes):
        if mode == "const":
            eps = eps_const
        else:
            eps = max(eps_min, eps_start * (eps_decay ** ep))
        hist_eps.append(eps)
        env = BallPursuitSimEnv(max_steps=40)
        state = env.reset()
        episode = []
        while True:
            if np.random.random() < eps:
                action = np.random.randint(n_actions)
            else:
                q_vals = Q[state]
                if q_vals[0] == q_vals[1] == q_vals[2] == q_vals[3]:
                    action = np.random.randint(n_actions)
                else:
                    action = int(np.argmax(q_vals))
            next_state, reward, done, _ = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if done:
                break
        G = 0.0
        visited = set()
        for (s, a, r) in reversed(episode):
            G = gamma * G + r
            if (s, a) not in visited:
                visited.add((s, a))
                returns_sum[s][a] += G
                returns_count[s][a] += 1
                Q[s][a] = returns_sum[s][a] / returns_count[s][a]
        g0 = sum([(gamma ** t) * r for t, (_, _, r) in enumerate(episode)])
        hist_g0.append(g0)
        hist_steps.append(len(episode))
        hist_succ.append(1 if episode[-1][2] > 50.0 else 0)
    return Q, hist_g0, hist_steps, hist_succ, hist_eps
print("Funcion de entrenamiento lista.")

## 4 Entrenamiento con epsilon constante

Epsilon fijo igual a 0.1. Mismo entorno, recompensa, gamma y episodios que la otra rama.

In [ ]:
t0 = time.time()
Q_const, g0_const, steps_const, succ_const, eps_const_hist = train_mc_control(n_episodes=3500, gamma=0.99, mode="const", eps_const=0.1, seed=42)
print(f"Constante listo en {time.time()-t0:.1f} s.")
print(f"G0 ultimos 100 {np.mean(g0_const[-100:]):.2f}.")
print(f"Exito ultimos 100 {np.mean(succ_const[-100:]):.3f}.")
print(f"Pasos ultimos 100 {np.mean(steps_const[-100:]):.2f}.")

## 5 Entrenamiento con epsilon decreciente geometrico

Epsilon por episodio igual a maximo entre 0.05 y 1.0 por 0.998 elevado al episodio. Mismo entorno, recompensa, gamma y episodios.

In [ ]:
t0 = time.time()
Q_dec, g0_dec, steps_dec, succ_dec, eps_dec_hist = train_mc_control(n_episodes=3500, gamma=0.99, mode="decay", eps_start=1.0, eps_min=0.05, eps_decay=0.998, seed=42)
print(f"Decreciente listo en {time.time()-t0:.1f} s.")
print(f"G0 ultimos 100 {np.mean(g0_dec[-100:]):.2f}.")
print(f"Exito ultimos 100 {np.mean(succ_dec[-100:]):.3f}.")
print(f"Pasos ultimos 100 {np.mean(steps_dec[-100:]):.2f}.")

## 6 Comparacion de G0

Curva de retorno descontado G0 con media movil de 100 episodios.

In [ ]:
window = 100
def media_movil(x, w):
    return np.convolve(np.asarray(x, float), np.ones(w) / w, mode="valid")
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(g0_const, alpha=0.12, color="#1f77b4")
ax.plot(g0_dec, alpha=0.12, color="#d62728")
ax.plot(range(window-1, len(g0_const)), media_movil(g0_const, window), color="#1f77b4", linewidth=2.0, label="Constante eps 0.1")
ax.plot(range(window-1, len(g0_dec)), media_movil(g0_dec, window), color="#d62728", linewidth=2.0, label="Decreciente")
ax.set_title("Retorno acumulado G0 por episodio")
ax.set_xlabel("Episodio")
ax.set_ylabel("G0 descontado gamma 0.99")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/learning_curves.png", dpi=150)
plt.show()

## 7 Comparacion de tasa de exito

Exito igual a 1 si hay captura. Media movil de 100 episodios.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(window-1, len(succ_const)), media_movil(succ_const, window), color="#1f77b4", linewidth=2.0, label="Constante eps 0.1")
ax.plot(range(window-1, len(succ_dec)), media_movil(succ_dec, window), color="#d62728", linewidth=2.0, label="Decreciente")
ax.set_title("Tasa de exito temporal")
ax.set_xlabel("Episodio")
ax.set_ylabel("Tasa de exito")
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/success_rate.png", dpi=150)
plt.show()
print(f"Exito final constante {np.mean(succ_const[-100:]):.3f}.")
print(f"Exito final decreciente {np.mean(succ_dec[-100:]):.3f}.")

## 8 Comparacion de pasos por episodio

Menos pasos indica intercepcion mas rapida. Media movil de 100.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps_const, alpha=0.12, color="#1f77b4")
ax.plot(steps_dec, alpha=0.12, color="#d62728")
ax.plot(range(window-1, len(steps_const)), media_movil(steps_const, window), color="#1f77b4", linewidth=2.0, label="Constante eps 0.1")
ax.plot(range(window-1, len(steps_dec)), media_movil(steps_dec, window), color="#d62728", linewidth=2.0, label="Decreciente")
ax.set_title("Pasos por episodio")
ax.set_xlabel("Episodio")
ax.set_ylabel("Pasos")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/steps_per_episode.png", dpi=150)
plt.show()
print(f"Pasos finales constante {np.mean(steps_const[-100:]):.2f}.")
print(f"Pasos finales decreciente {np.mean(steps_dec[-100:]):.2f}.")

## Metricas en CSV

Se guardan episodio, estrategia, retorno, exito, pasos y epsilon para reproducir las curvas.

In [ ]:
import csv
path = f"{OUT}/metrics.csv"
with open(path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["episode", "exploration_strategy", "return", "success", "steps", "epsilon"])
    for ep in range(len(g0_const)):
        w.writerow([ep, "constante", f"{g0_const[ep]:.4f}", succ_const[ep], steps_const[ep], f"{eps_const_hist[ep]:.6f}"])
    for ep in range(len(g0_dec)):
        w.writerow([ep, "decreciente", f"{g0_dec[ep]:.4f}", succ_dec[ep], steps_dec[ep], f"{eps_dec_hist[ep]:.6f}"])
print(f"CSV guardado en {path}.")

## 9 Funcion de valor V y politica pi

Se usa la tabla de la rama decreciente porque alcanza el criterio de exito. V es maximo de Q. Pi es argmax de Q. Frontal pide DASH fuerte. Derecha pide giro derecha. Izquierda pide giro izquierda.

In [ ]:
Q_opt = Q_dec
Q_tabla = np.array([[Q_opt[(d, a)] for a in range(5)] for d in range(4)])
np.save(os.path.join(OUT, "q_policy.npy"), Q_tabla)
dist_labels = ["Zona 0 (<0.8m)", "Zona 1 (0.8-3m)", "Zona 2 (3-8m)", "Zona 3 (>8m)"]
angle_labels = ["Frontal", "Der (-60 a -15)", "Izq (15 a 60)", "Atras Der", "Atras Izq"]
action_abbr = ["DASH 100", "DASH 50", "GIRAR IZQ", "GIRAR DER"]
policy_grid = np.zeros((4, 5))
q_max_grid = np.zeros((4, 5))
for d_idx in range(4):
    for a_idx in range(5):
        s = (d_idx, a_idx)
        policy_grid[d_idx, a_idx] = float(np.argmax(Q_opt[s]))
        q_max_grid[d_idx, a_idx] = float(np.max(Q_opt[s]))
fig, (ax_q, ax_pi) = plt.subplots(1, 2, figsize=(14, 5.0))
im_q = ax_q.imshow(q_max_grid, cmap="viridis", aspect="auto")
ax_q.set_xticks(range(5))
ax_q.set_xticklabels(angle_labels, rotation=20, ha="right", fontsize=9)
ax_q.set_yticks(range(4))
ax_q.set_yticklabels(dist_labels, fontsize=9)
ax_q.set_title("V(s) = max Q(s,a)")
fig.colorbar(im_q, ax=ax_q, shrink=0.7)
for i in range(4):
    for j in range(5):
        ax_q.text(j, i, f"{q_max_grid[i, j]:.1f}", ha="center", va="center", color="white", fontsize=9)
im_pi = ax_pi.imshow(policy_grid, cmap="tab10", vmin=0, vmax=9, aspect="auto")
ax_pi.set_xticks(range(5))
ax_pi.set_xticklabels(angle_labels, rotation=20, ha="right", fontsize=9)
ax_pi.set_yticks(range(4))
ax_pi.set_yticklabels(dist_labels, fontsize=9)
ax_pi.set_title("pi(s) = argmax Q(s,a)")
for i in range(4):
    for j in range(5):
        ax_pi.text(j, i, action_abbr[int(policy_grid[i, j])], ha="center", va="center", color="black", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/policy_value.png", dpi=150)
plt.show()

## 10 Trayectoria 2D sobre la cancha

Comparacion entre agente aleatorio y agente entrenado con la misma salida. Se guardan inicio, trayectoria, balon y punto final.

In [ ]:
def draw_pitch(ax, title):
    ax.set_facecolor("#2e7d32")
    ax.add_patch(patches.Rectangle((-52.5, -34), 105, 68, linewidth=1.8, edgecolor="white", facecolor="none"))
    ax.add_line(plt.Line2D([0, 0], [-34, 34], color="white", linewidth=1.8))
    ax.add_patch(patches.Circle((0, 0), 9.15, linewidth=1.8, edgecolor="white", facecolor="none"))
    ax.add_patch(patches.Circle((0, 0), 0.5, color="white"))
    ax.add_patch(patches.Rectangle((-52.5, -20.16), 16.5, 40.32, linewidth=1.2, edgecolor="white", facecolor="none"))
    ax.add_patch(patches.Rectangle((36.0, -20.16), 16.5, 40.32, linewidth=1.2, edgecolor="white", facecolor="none"))
    ax.set_xlim(-30, 15)
    ax.set_ylim(-18, 18)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_title(title)
np.random.seed(7)
env_eval = BallPursuitSimEnv(max_steps=40)
env_eval.player_x, env_eval.player_y, env_eval.player_theta = -15.0, 3.0, -120.0
env_eval.ball_x, env_eval.ball_y = 0.0, 0.0
env_eval.trajectory_x = [env_eval.player_x]
env_eval.trajectory_y = [env_eval.player_y]
env_eval.steps = 0
for _ in range(env_eval.max_steps):
    _, _, done, _ = env_eval.step(np.random.randint(4))
    if done:
        break
traj_rand = (list(env_eval.trajectory_x), list(env_eval.trajectory_y))
env_eval.player_x, env_eval.player_y, env_eval.player_theta = -15.0, 3.0, -120.0
env_eval.ball_x, env_eval.ball_y = 0.0, 0.0
env_eval.trajectory_x = [env_eval.player_x]
env_eval.trajectory_y = [env_eval.player_y]
env_eval.steps = 0
state = env_eval._get_state()
for _ in range(env_eval.max_steps):
    action = int(np.argmax(Q_opt[state]))
    state, _, done, _ = env_eval.step(action)
    if done:
        break
traj_opt = (list(env_eval.trajectory_x), list(env_eval.trajectory_y))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
draw_pitch(ax1, "Agente no entrenado (aleatorio)")
ax1.plot(traj_rand[0], traj_rand[1], "-o", color="#ff7f0e", markersize=3, linewidth=1.5, label="Trayectoria")
ax1.plot(traj_rand[0][0], traj_rand[1][0], "s", color="#1f77b4", markersize=8, label="Inicio")
ax1.plot(0, 0, "o", color="white", markeredgecolor="black", markersize=10, label="Balon")
ax1.legend(fontsize=8)
draw_pitch(ax2, "Agente entrenado (Monte Carlo)")
ax2.plot(traj_opt[0], traj_opt[1], "-o", color="#00e676", markersize=3, linewidth=2.0, label="Trayectoria")
ax2.plot(traj_opt[0][0], traj_opt[1][0], "s", color="#1f77b4", markersize=8, label="Inicio")
ax2.plot(0, 0, "o", color="white", markeredgecolor="black", markersize=10, label="Balon")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/trajectory.png", dpi=150)
plt.show()
print(f"Pasos aleatorio {len(traj_rand[0])-1}. Pasos entrenado {len(traj_opt[0])-1}.")

## 11 Conexion con el servidor RoboCup 2D

Se usa el cliente del starter sin cambios. Reiniciar robocup_server antes de ejecutar el cuaderno para que move ubique al jugador en (-15, 0). El servidor tarda cerca de 30 segundos en comenzar el partido. La grafica muestra la distancia recibida del servidor; captura cuando baja de 0.8 m.

In [ ]:
try:
    from robocup_client import RoboCup2DClient

    host = os.getenv("SERVER_HOST", "rcssserver")
    port = int(os.getenv("SERVER_PORT", "6000"))
    ruta_q = os.path.join(OUT, "q_policy.npy")
    if "Q_opt" not in globals() and os.path.exists(ruta_q):
        Q_opt = np.load(ruta_q)
    if "Q_opt" not in globals():
        raise RuntimeError("Corre primero las celdas de entrenamiento.")
    if "action_abbr" not in globals():
        action_abbr = ["DASH 100", "DASH 50", "GIRAR IZQ", "GIRAR DER"]
    env_live = BallPursuitSimEnv()

    client = RoboCup2DClient(host=host, port=port, team_name="UTEC_MC_Opt")
    if not client.connect(init_pos=(-15.0, 0.0)):
        raise RuntimeError("Sin conexion con rcssserver.")

    print("Conectado. Esperando a que arranque el partido (before_kick_off)...")
    t0 = time.time()
    reloj = -1
    while time.time() - t0 < 45:
        obs = client.get_latest_observation()
        if obs["time"] != reloj:
            reloj = obs["time"]
            if reloj >= 2:
                break
        time.sleep(0.5)
    if reloj < 2:
        raise RuntimeError("El servidor no arranca el partido. Ejecuta docker restart robocup_server.")
    ball0 = client.get_latest_observation()["ball"]
    if ball0:
        print(f"Balon visto a {ball0[0]:.1f} m desde (-15, 0). Se ejecuta pi en vivo (hasta 100 pasos).")

    dists = []
    captura = None
    for paso in range(100):
        obs = client.get_latest_observation()
        ball = obs["ball"]
        if ball is None:
            client.turn(moment=45.0)
            time.sleep(0.1)
            continue
        dist, angle = ball
        s = env_live._discretize(dist, angle)
        best = int(np.argmax(Q_opt[s]))
        if best == 0:
            client.dash(power=100.0)
        elif best == 1:
            client.dash(power=50.0)
        elif best == 2:
            client.turn(moment=35.0)
        else:
            client.turn(moment=-35.0)
        dists.append(dist)
        if paso % 5 == 0 or dist < 1.0:
            print(f"Paso {paso:03d} | t={obs['time']:4d} | dist={dist:5.2f} | {action_abbr[best]}")
        if dist < 0.8:
            captura = paso
            break
        time.sleep(0.1)
    client.close()

    if captura is None:
        print("Sin captura dentro del presupuesto en vivo.")
    else:
        print(f"Captura en vivo en el paso {captura} contra el servidor real.")

    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(range(len(dists)), dists, marker="o", ms=3)
    ax.axhline(0.8, ls="--", color="crimson", label="Umbral de captura (0.8 m)")
    ax.set_xlabel("Paso")
    ax.set_ylabel("Distancia al balon (m)")
    ax.set_title("Politica aprendida en vivo contra rcssserver")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Prueba en vivo omitida. Motivo {e}")


## 12 Conclusiones

La rama decreciente alcanza 100 por ciento de exito en menos de 40 pasos con G0 cerca de 85. La rama constante con epsilon 0.1 se queda en 0 por ciento con G0 cerca de menos 9. El decaimiento geometrico explora al inicio y explota al final. Epsilon fijo bajo se atasca porque no visita la captura. La politica final gira hacia el balon y avanza con DASH 100 de frente. El criterio del PDF se cumple con la politica decreciente.